# 映画のおすすめ: 検索（Retrieval）

**学習目標**

このノートブックでは、MovieLens データセットを使用して 2 タワー（two-tower）検索モデルを構築およびトレーニングします。

具体的には以下を行います。

1. データを取得し、トレーニングセットとテストセットに分割する。
2. 検索モデルを実装する。
3. モデルを適合（fit）および評価する。
4. 近似最近傍（ANN）インデックスを構築して、効率的なサービング用にエクスポートする。

## はじめに
現実世界のレコメンダーシステムは、多くの場合 2 つのステージで構成されます。

1. **検索（Retrieval）ステージ**: すべての候補の中から、最初の数百個の候補セットを選択する役割を持ちます。このモデルの主な目的は、ユーザーが興味のないすべての候補を効率的に排除することです。検索モデルは何百万もの候補を扱う可能性があるため、計算効率が高くなければなりません。
2. **ランキング（Ranking）ステージ**: 検索モデルの出力を受け取り、それらを微調整して最適な一握りのおすすめを選択します。その役割は、ユーザーが興味を持つ可能性のあるアイテムのセットを、見込みのある候補のショートリストに絞り込むことです。

このノートブックでは、最初のステージである検索に焦点を当てます。

検索モデルは多くの場合、2 つのサブモデルで構成されます。

1. **クエリモデル**: クエリ特徴量を使用してクエリ表現（通常は固定次元の埋め込みベクトル）を計算します。
2. **候補モデル**: 候補特徴量を使用して候補表現（同じサイズのベクトル）を計算します。

これら 2 つのモデルの出力を乗算してクエリと候補の親和性スコアを算出し、スコアが高いほど候補とクエリの間のマッチ度が高いことを表します。

この 2 つのサブモデルからなるアーキテクチャは **2 タワー（two-tower）** モデルと呼ばれます。

### データセット

MovieLens データセットは、ミネソタ大学の [GroupLens](https://grouplens.org/datasets/movielens/) 研究グループによって作成されたクラシックなデータセットです。ユーザーのセットによって映画に付けられた一連の評価が含まれており、推薦システム研究の主力データセットとなっています。

このデータは 2 つの方法で扱うことができます。

1. ユーザーがどの映画を視聴（および評価）し、どの映画を視聴しなかったかを表していると解釈できます。これは暗黙的フィードバック（implicit feedback）の一形態であり、ユーザーの視聴履歴から、ユーザーが好んで見たいものと見たくないものがわかります。
2. ユーザーが実際に視聴した映画をどの程度気に入ったかを表していると見ることもできます。これは明示的フィードバック（explicit feedback）の一形態です。ユーザーが映画を視聴したと仮定すると、付けられた評価を見ることでどの程度気に入ったかを大まかに知ることができます。

このノートブックでは、カタログからユーザーが見る可能性の高い映画のセットを予測するモデルである検索システムに焦点を当てます。多くの場合、ここでは暗黙的データの方が有用であるため、MovieLens を暗黙的システムとして扱います。つまり、ユーザーが視聴したすべての映画を正例とし、まだ見ていないすべての映画を暗黙的な負例とみなします。

各学習目標は、[演習用ラボノートブック](https://github.com/GoogleCloudPlatform/training-data-analyst/blob/master/courses/machine_learning/deepdive2/recommendation_systems/solutions/basic_retrieval.ipynb) 内の __#TODO__ に対応しています。この解答ノートブックを確認する前に、まずは演習用ノートブックを完了してみてください。

## セットアップ

In [ ]:
# 必要なモジュールのインポート
import os
import warnings

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings("ignore")

In [ ]:
import pprint
import tempfile
from typing import Dict, Text

import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds

In [ ]:
import tensorflow_recommenders as tfrs

## データセットの準備

まずはデータを確認してみましょう。

[TensorFlow Datasets](https://www.tensorflow.org/datasets) の MovieLens データセットを使用します。`movielens/100k-ratings` を読み込むと評価データを含む `tf.data.Dataset` オブジェクトが生成され、`movielens/100k-movies` を読み込むと映画データのみを含む `tf.data.Dataset` オブジェクトが生成されます。

MovieLens データセットには事前定義された分割がないため、すべてのデータが `train` スプリットに含まれていることに注意してください。

In [ ]:
# TODO 1 - ここにコードを記述してください。
# 評価データ。
ratings = tfds.load(
    "movielens/100k-ratings",
    split="train",
    data_dir="gs://asl-public-data/data/tensorflow_datasets/",
)
# 利用可能なすべての映画の特徴量。
movies = tfds.load(
    "movielens/100k-movies",
    split="train",
    data_dir="gs://asl-public-data/data/tensorflow_datasets/",
)

ratings データセットは、映画 ID、ユーザー ID、割り当てられた評価、タイムスタンプ、映画情報、ユーザー情報の辞書を返します。

In [ ]:
# ユーザー情報と映画情報を表示
for x in ratings.take(1).as_numpy_iterator():
    pprint.pprint(x)

movies データセットには、映画 ID、映画のタイトル、およびそれが属するジャンルに関するデータが含まれています。ジャンルは整数のラベルでエンコードされていることに注意してください。

In [ ]:
# どのジャンルに属しているかに関するデータを表示
for x in movies.take(1).as_numpy_iterator():
    pprint.pprint(x)

この例では、ratings データに焦点を当てます。他のノートブックでは、モデルの品質を向上させるために映画情報データも使用する方法を検討します。

データセットには `user_id` と `movie_title` フィールドのみを保持します。

In [ ]:
# ここでは評価データに焦点を当てます
ratings = ratings.map(
    lambda x: {
        "movie_title": x["movie_title"],
        "user_id": x["user_id"],
    }
)
movies = movies.map(lambda x: x["movie_title"])

モデルの適合（fit）と評価を行うには、トレーニングセットと評価セットに分割する必要があります。産業用のレコメンダーシステムでは、多くの場合時間によって分割されます。つまり、時刻 $T$ までのデータを使用して時刻 $T$ 以降のインタラクションを予測します。


ただし、このシンプルな例ではランダム分割を使用し、評価の 80% をトレーニングセットに、20% をテストセットに割り当てます。

In [ ]:
# ここでは、tf.random モジュールを使用して、最初の次元でテンソルをランダムにシャッフルします
SEED = 42

tf.random.set_seed(SEED)
shuffled = ratings.shuffle(100_000, seed=SEED, reshuffle_each_iteration=False)

train = shuffled.take(80_000)
test = shuffled.skip(80_000).take(20_000)

データに存在する一意のユーザー ID と映画のタイトルも把握しておきましょう。 

これは、カテゴリ特徴量の未加工の値をモデル内の埋め込みベクトルにマッピングできるようにする必要があるため重要です。そのためには、未加工の特徴量の値を連続した範囲の整数にマッピングする語彙（ボキャブラリ）が必要です。これにより、埋め込みテーブル内の対応する埋め込みを検索できるようになります。

In [ ]:
# 埋め込みテーブルに従って対応するデータを表示。
movie_titles = movies.batch(1_000)
user_ids = ratings.batch(1_000_000).map(lambda x: x["user_id"])

unique_movie_titles = np.unique(np.concatenate(list(movie_titles)))
unique_user_ids = np.unique(np.concatenate(list(user_ids)))

unique_movie_titles[:10]

## モデルの実装

モデルのアーキテクチャの選択は、モデリングの重要な部分です。

ここでは 2 タワー検索モデルを構築するため、各タワーを個別に構築してから、最終的なモデルに統合することができます。

### クエリタワー

まずはクエリタワーから始めましょう。

最初のステップは、クエリと候補の表現の次元数を決定することです。

In [ ]:
embedding_dimension = 32

値を大きくすると、より正確になる可能性があるモデルに対応しますが、適合が遅くなり、過学習が発生しやすくなります。

2 つ目のステップは、モデル自体を定義することです。ここでは、Keras の前処理レイヤーを使用して、まずユーザー ID を整数に変換し、次に `Embedding` レイヤーを介してそれらをユーザーの埋め込みに変換します。先ほど計算した一意のユーザー ID のリストを語彙として使用することに注意してください。

In [ ]:
user_model = tf.keras.Sequential(
    [
        tf.keras.layers.experimental.preprocessing.StringLookup(
            vocabulary=unique_user_ids, mask_token=None
        ),
        # 未知のトークンを考慮して追加の埋め込みを追加します。
        tf.keras.layers.Embedding(
            len(unique_user_ids) + 1, embedding_dimension
        ),
    ]
)

この単純なモデルに対して `tf.keras.Model` のサブクラスを定義するのは過剰かもしれませんが、最終的に `embedding_dimension` 幅の出力を返す限り、標準の Keras コンポーネントを使用して任意の複雑なモデルに簡単に拡張できます。

### 候補タワー

候補タワーに対しても同様の処理を行うことができます。

In [ ]:
movie_model = tf.keras.Sequential(
    [
        tf.keras.layers.experimental.preprocessing.StringLookup(
            vocabulary=unique_movie_titles, mask_token=None
        ),
        tf.keras.layers.Embedding(
            len(unique_movie_titles) + 1, embedding_dimension
        ),
    ]
)

### 指標（Metrics）

トレーニングデータには、正例の（ユーザー, 映画）ペア、つまり実際の評価が存在するペアがあります。モデルの性能を把握するには、モデルがこのペアに対して計算する親和性スコアを、他のすべての候補（すべての映画）のスコアと比較する必要があります。正例のペアの分類スコアが他のすべての候補よりも高ければ、モデルの精度が高いことがわかります。

これを行うには、`tfrs.metrics.FactorizedTopK` 指標を使用できます。この指標には必須の引数が 1 つあります。評価用の暗黙的な負例（明示的な評価がない映画候補）として使用される候補のデータセットです。
今回の場合、それは映画モデルを介して埋め込みに変換された `movies` データセットです。

`FactorizedTopK` は複数の指標を自動的に計算します。デフォルトでは上位 [1, 5, 10, 50, 100] の [`TopKCategoricalAccuracy`](https://www.tensorflow.org/api_docs/python/tf/keras/metrics/TopKCategoricalAccuracy) です。<br>
Top K 精度は、すべての負例候補（この場合はすべての映画）の中から正例ペアが上位 K 個以内に正しくランク付けされた割合を表します。

In [ ]:
# ここで、tfrs.metrics.FactorizedTopK 関数は、検索モデルによって提示された上位 K 個の候補全体の指標を計算します。
metrics = tfrs.metrics.FactorizedTopK(
    candidates=movies.batch(128).map(movie_model)
)

### 損失（Loss）

次のコンポーネントは、モデルのトレーニングに使用される損失です。TFRS には、これを簡単にするためのいくつかの損失レイヤーとタスクが用意されています。

ここでは、損失関数と指標の計算をまとめた便利なラッパーである `tfrs.tasks.Retrieval` タスクオブジェクトを使用します。

内部的には、`Retrieval` タスクレイヤーはクエリの埋め込みと候補の埋め込みを乗算してスコアを計算し、損失（デフォルトではクロスエントロピー損失）を出力します。<br>
必要に応じて、指標を提供してトレーニング中に計算することもできます。

In [ ]:
# TODO 2 - ここにコードを記述してください。
# ここで、関数は損失関数と指標の計算をまとめます。
task = tfrs.tasks.Retrieval(metrics=metrics)

タスク自体は、クエリと候補の埋め込みを引数として受け取り、計算された損失を返す Keras レイヤーです。これを使用して、モデルのトレーニングループを実装します。

### モデル全体の構築

これらすべてをまとめて 1 つのモデルに統合します。TFRS はモデル構築を効率化するベースモデルクラス（`tfrs.models.Model`）を提供しています。必要なのは、`__init__` メソッドでコンポーネントをセットアップし、未加工の特徴量を受け取って損失値を返す `compute_loss` メソッドを実装することだけです。

ベースモデルが、モデルに適合させるための適切なトレーニングループの作成を自動的に処理します。

In [ ]:
class MovielensModel(tfrs.Model):
    def __init__(self, user_model, movie_model):
        super().__init__()
        self.movie_model: tf.keras.Model = movie_model
        self.user_model: tf.keras.Model = user_model
        self.task: tf.keras.layers.Layer = task

    def compute_loss(self, features, training=False):
        # ユーザー特徴量を取り出し、ユーザーモデルに渡します。
        user_embeddings = self.user_model(features["user_id"])
        # そして映画の特徴量を取り出し、映画モデルに渡して、
        # 埋め込みを取得します。
        positive_movie_embeddings = self.movie_model(features["movie_title"])

        # タスクが損失と指標を計算します。
        return self.task(user_embeddings, positive_movie_embeddings)

`tfrs.Model` ベースクラスはシンプルで便利です。同じメソッドを使用してトレーニング損失とテスト損失の両方を計算できます。

内部的には、依然としてプレーンな Keras モデルです。次のように `tf.keras.Model` を継承し、`train_step` および `test_step` 関数をオーバーライドすることで、同じ機能を実現できます（詳細については[ガイド](https://www.tensorflow.org/guide/keras/customizing_what_happens_in_fit)を参照してください）。


```python
class MovielensModel(tf.keras.Model):
    def __init__(self, user_model, movie_model):
        super().__init__()
        self.movie_model: tf.keras.Model = movie_model
        self.user_model: tf.keras.Model = user_model
        self.task: tf.keras.layers.Layer = task

    def train_step(self, features):

        # 勾配を記録するための gradient tape をセットアップ。
        with tf.GradientTape() as tape:

            # 損失の計算。
            user_embeddings = self.user_model(features["user_id"])
            positive_movie_embeddings = self.movie_model(
                features["movie_title"]
            )
            loss = self.task(user_embeddings, positive_movie_embeddings)

            # 正則化損失も処理。
            regularization_loss = sum(self.losses)

            total_loss = loss + regularization_loss

        gradients = tape.gradient(total_loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(gradients, self.trainable_variables))

        metrics = {metric.name: metric.result() for metric in self.metrics}
        metrics["loss"] = loss
        metrics["regularization_loss"] = regularization_loss
        metrics["total_loss"] = total_loss

        return metrics

    def test_step(self, features):

        # 損失の計算。
        user_embeddings = self.user_model(features["user_id"])
        positive_movie_embeddings = self.movie_model(features["movie_title"])
        loss = self.task(user_embeddings, positive_movie_embeddings)

        # 正則化損失も処理。
        regularization_loss = sum(self.losses)

        total_loss = loss + regularization_loss

        metrics = {metric.name: metric.result() for metric in self.metrics}
        metrics["loss"] = loss
        metrics["regularization_loss"] = regularization_loss
        metrics["total_loss"] = total_loss

        return metrics
```

ただし、このノートブックではモデリングに集中しボイラープレートコードを抽象化するために、`tfrs.Model` ベースクラスを使用し続けます。

## 適合（Fitting）と評価

モデルを定義したら、標準の Keras の fit および evaluate ルーチンを使用してモデルを適合および評価できます。

まずモデルをインスタンス化しましょう。

In [ ]:
# モデルのコンパイル。
model = MovielensModel(user_model, movie_model)
model.compile(optimizer=tf.keras.optimizers.Adagrad(learning_rate=0.1))

次に、トレーニングデータと評価データをシャッフル、バッチ処理、キャッシュします。

In [ ]:
cached_train = train.shuffle(100_000).batch(8192).cache()
cached_test = test.batch(4096).cache()

次にモデルをトレーニングします。

In [ ]:
# TODO 3a - ここにコードを記述してください。
# モデルのトレーニング。
model.fit(cached_train, epochs=3)

モデルのトレーニングが進むにつれて損失が減少し、一連の top-k 検索指標が更新されます。これらは、真の正例が候補セット全体から取得された上位 k 個のアイテムに含まれているかどうかを示します。例えば、top-5 カテゴリ精度指標が 0.2 の場合、平均して 20% の確率で真の正例が取得された上位 5 個のアイテムに含まれていることを示します。

なお、この例では評価時だけでなくトレーニング中にも指標を評価していることに注意してください。大規模な候補セットでは処理がかなり遅くなる可能性があるため、トレーニング中は指標計算をオフにして評価時にのみ実行するのが賢明な場合があります。

最後に、テストセットでモデルを評価します。

In [ ]:
# TODO 3b - ここにコードを記述してください。
# モデルの評価。
model.evaluate(cached_test, return_dict=True)

テストセットでのパフォーマンスは、トレーニング時のパフォーマンスよりも大幅に低下しています。これは主に 2 つの要因によるものです。

1. モデルは一度確認したデータを単に記憶できるため、そのデータに対してより高いパフォーマンスを発揮する傾向があります。この過学習現象は、モデルのパラメータ数が多い場合に特に顕著になります。これは、モデルの正則化や、モデルが未知のデータに対してより適切に汎化できるようにするユーザー特徴量や映画特徴量の使用によって緩和できます。
2. モデルがユーザーの視聴済み映画の一部を再度おすすめしていることです。これらの既知の正例である視聴履歴が、テスト対象の映画を上位 K 個のおすすめから締め出してしまう可能性があります。

2 番目の現象は、過去に視聴した映画をテストおすすめから除外することで対処できます。このアプローチは推薦システムの文献では比較的一般的ですが、このノートブックでは採用していません。過去の視聴履歴をおすすめしないことが重要である場合、適切に指定されたモデルが過去のユーザー履歴とコンテキスト情報からこの動作を自動的に学習することが期待されます。さらに、同じアイテムを複数回おすすめすることが適切な場合もよくあります（例えば、定番のテレビシリーズや定期的に購入される商品など）。

## 予測の実行

モデルが完成したので、予測を実行できるようにします。これを行うには、`tfrs.layers.factorized_top_k.BruteForce` レイヤーを使用できます。

In [ ]:
# 未加工のクエリ特徴量を受け取り、
index = tfrs.layers.factorized_top_k.BruteForce(model.user_model)
# 映画データセット全体から映画を推薦するモデルを作成します。
index.index_from_dataset(
    tf.data.Dataset.zip(
        (movies.batch(100), movies.batch(100).map(model.movie_model))
    )
)

# おすすめを取得。
_, titles = index(tf.constant(["42"]))
print(f"Recommendations for user 42: {titles[0, :3]}")

もちろん、多数の候補が存在するモデルを提供する場合、`BruteForce` レイヤーは遅すぎます。以下のセクションでは、近似検索インデックスを使用してこれを高速化する方法を示します。

## モデルのサービング

モデルをトレーニングしたら、それをデプロイする方法が必要です。

2 タワー検索モデルにおいて、サービングには 2 つのコンポーネントがあります。

- **サービングクエリモデル**: クエリの特徴量を受け取り、それらをクエリの埋め込みに変換します。
- **サービング候補モデル**: 多くの場合、クエリモデルによって生成されたクエリに応じて候補を高速に近似検索できる、近似最近傍（ANN）インデックスの形式を取ります。

TFRS では、両方のコンポーネントを 1 つのエクスポート可能なモデルにパッケージ化でき、未加工のユーザー ID を受け取ってそのユーザーに対する上位の映画タイトルを返すモデルが得られます。これはモデルを `SavedModel` 形式にエクスポートすることで実現され、[TensorFlow Serving](https://www.tensorflow.org/tfx/guide/serving) を使用したサービングが可能になります。

このようなモデルをデプロイするには、上記で作成した `BruteForce` レイヤーをエクスポートするだけです。

In [ ]:
# TODO 4 - ここにコードを記述してください。
# クエリモデルをエクスポート。
with tempfile.TemporaryDirectory() as tmp:
    path = os.path.join(tmp, "model")

    # インデックスを保存。
    tf.saved_model.save(index, path)

    # 再読み込み。TensorFlow Serving でも実行可能です。
    loaded = tf.saved_model.load(path)

    # ユーザー ID を渡して、上位の予測映画タイトルを取得。
    scores, titles = loaded(["42"])

    print(f"Recommendations: {titles[0][:3]}")

また、予測を高速化するために近似検索インデックスをエクスポートすることもできます。これにより、数千万件の候補セットからおすすめを効率的に提示することが可能になります。

高速な近似検索モデルの使用と調整の詳細については、[効率的なサービング（efficient serving）](https://tensorflow.org/recommenders/examples/efficient_serving) ノートブックをご覧ください。

## 次のステップ

これで検索に関するノートブックは終了です。

ここで紹介した内容を発展させるには、以下をご確認ください。

1. マルチタスクモデルの学習: 評価とクリック数を共同で最適化する。
2. 映画メタデータの活用: コールドスタート問題を軽減するために、より複雑な映画モデルを構築する。